# Paquetes de Python

In [4]:
import numpy as np
from typing import Dict
from typing import Optional
from typing import Tuple

ModuleNotFoundError: No module named 'numpy'

# 2. Linear Layer Backward

In [ ]:
def linear_backward(dout: np.ndarray, x: np.ndarray, w: np.ndarray, b: np.ndarray) -> Dict[str, np.ndarray]:
    """
    Computes dx, dw, db for y = x @ w + b.
    
    Args:
        dout: Upstream gradient (N, Dout)
        x: Input (N, Din)
        w: Weights (Din, Dout)
        b: Bias (Dout,)
        
    Returns:
        Dict with "dx", "dw", "db"
    """
    dx =  dout @ w.T
    dw = x.T @ dout
    db = dout.sum(axis=0)
    return {"dx":dx, "dw":dw, "db":db}
    pass

## Contexto

En una capa lineal de una red neuronal, la operación directa se define como:

$$
y = x \cdot W + b
$$

donde:
- $x \in \mathbb{R}^{N \times D_{in}}$: matriz de entrada
- $W \in \mathbb{R}^{D_{in} \times D_{out}}$: matriz de pesos
- $b \in \mathbb{R}^{D_{out}}$: vector de sesgos
- $y \in \mathbb{R}^{N \times D_{out}}$: salida de la capa

Durante el **backpropagation**, se busca calcular los gradientes de la función de pérdida L respecto a cada parámetro: x, W y b.

---

## Derivadas parciales

### 1️. Gradiente respecto a la entrada x
Aplicando la regla de la cadena:

$$
\frac{\partial L}{\partial x} = \frac{\partial L}{\partial y} \cdot \frac{\partial y}{\partial x}
$$

Como $\dfrac{\partial y}{\partial x} = W$, se obtiene:

$$
dx = dout \cdot W^T
$$

donde `dout` representa el gradiente que proviene de la capa siguiente $\frac{\partial L}{\partial y}$.

En código:
```python
dx =  dout @ w.T
```

---

### 2️. Gradiente respecto a los pesos W
De forma análoga:

$$
\frac{\partial L}{\partial W} = x^T \cdot \frac{\partial L}{\partial y}
$$

por lo tanto:

$$
dw = x^T \cdot dout
$$

En código:
```python
dw = x.T @ dout
```

---

### 3️. Gradiente respecto al sesgo b
El sesgo se suma a cada fila de la salida, por lo que su gradiente es la suma de los gradientes de salida a lo largo del batch:

$$
db = \sum_{i=1}^{N} dout_i
$$

En código:  
```python
db = dout.sum(axis=0)
```

#4. ReLU Activation Backward


In [ ]:

def relu_backward(dout: np.ndarray, x: np.ndarray) -> np.ndarray:
   

    # La derivada de ReLU es, a través de una indicadora:
    # 1 si x > 0
    # 0 si x <= 0
    mask = x > 0

    # Aplicamos regla de la cadena.
    dx = dout * mask

    return dx 

## Explicación
Nos interesa saber que tanto cambia el gradiente, si la entrada es positiva , 
se conserva el gradiente pero si es cero o negativa se anula, así vemos que
neuronas aportaron al resultado y cuales no, en el ajuste de parámetros.

# 5 Sigmoid Activation

In [ ]:
def sigmoid_ops(x: np.ndarray, dout: np.ndarray) -> Dict[str, np.ndarray]:
  
    # Forward:
    # La sigmoide transforma cualquier número real en un valor entre 0 y 1.
    # Usamos np.where para evitar problemas numéricos con valores muy grandes.
    out = np.where(
        x >= 0,
        1 / (1 + np.exp(-x)),
        np.exp(x) / (1 + np.exp(x))
    )

    # Backward:
    # La derivada de sigmoid(x) es sigmoid(x) * (1 - sigmoid(x)).
    sigmoid_prime = out * (1 - out)

    # Regla de la cadena.
    dx = dout * sigmoid_prime

    return {
        "out": out,
        "dx": dx
    }



Se usa la función sigmoide tanto atrás como adelante, que se basa en agarrar
 cualquier valor real y mandarlo a un punto entre cero y uno, se defino como
 1 / (1 +np.exp(-x)). Cuando x es muy grande el resultado se acerca a 1 y cuando
 es negativo el resultado se acerca a 0. y después se ve en la parte de 
 backward que se usa paraver como se propaga la gradiente a través del 
 sigmoide.

# 6. Tanh Activation

In [ ]:
def tanh_ops(x: np.ndarray, dout: np.ndarray) -> Dict[str, np.ndarray]:
   """
   Computes tanh forward and backward.
   """
   out = np.tanh(x)
   tanh_prime = 1 - np.tanh(x)**2
   dx = dout*tanh_prime
   return {"out":out, "dx":dx}
   pass

## Contexto


La función de activación tangente hiperbólica se define como:


$$
\tanh(x) = \frac{\sinh(x)}{\cosh(x)} = \frac{e^x - e^{-x}}{e^x + e^{-x}}
$$


Su rango es \(-1, 1\), y se utiliza para normalizar valores en redes neuronales.


---


## Derivación del gradiente
Para el **backward pass**, necesitamos la derivada de $\tanh(x)$ respecto a x:


$$
\frac{d}{dx}\tanh(x) = 1 - \tanh^2(x)
$$


Esto se obtiene aplicando la regla del cociente o usando la identidad hiperbólica:


$$
\cosh^2(x) - \sinh^2(x) = 1
$$


Por tanto:


$$
\frac{d}{dx}\tanh(x) = \frac{\cosh^2(x) - \sinh^2(x)}{\cosh^2(x)} = 1 - \tanh^2(x)
$$


---


## Interpretación en el backward pass
Durante la propagación hacia atrás, el gradiente de la pérdida L respecto a la entrada x se calcula como:


$$
dx = dout \cdot (1 - \tanh^2(x))
$$


donde `dout` es el gradiente que proviene de la capa siguiente.

# 7. MSE Loss

In [ ]:
def mse_loss(y_pred: np.ndarray, y_true: np.ndarray) -> Dict[str, float | np.ndarray]:


    # Diferencia entre la predicción y el valor real.
    error = y_pred - y_true

    # MSE: promedio de los errores al cuadrado.
    loss = np.mean(error ** 2)

    # Número total de elementos.
    num_elements = y_pred.size

    # Gradiente del MSE respecto a y_pred.
    dx = (2 / num_elements) * error

    return {
        "loss": loss,
        "dx": dx
    }

En este ejercicio se implementa la función de pérdida de error cuadrático 
medio. Esta función mide qué tan alejadas están las predicciones del modelo 
respecto a los valores reales. Primero se calcula la diferencia entre la
 predicción y el valor verdadero, luego se eleva al cuadrado y finalmente se
 obtiene el promedio de todos los errores. Además, se calcula el gradiente 
 respecto a las predicciones, el cual indica cómo debe ajustarse la salida
 del modelo para reducir la pérdida. Este gradiente es necesario para la 
 retropropagación, ya que permite actualizar los parámetros de la red neuronal
 en la dirección que disminuye el error.

# 10. SGD Optimizer

In [ ]:
def sgd_step(w: np.ndarray, dw: np.ndarray, lr: float) -> np.ndarray:
    """
    Updates w using SGD.
    """
    step = lr*dw
    w_new = w - step

    return w_new
    pass

## Contexto

El **Descenso de Gradiente Estocástico (SGD)** es el algoritmo de optimización más básico y fundamental para entrenar redes neuronales.  
Su objetivo es minimizar una función de pérdida L(w) ajustando los parámetros w en la dirección opuesta al gradiente.

---

## Derivación matemática

La actualización de los parámetros se define como:

$$
w_{\text{new}} = w - \eta \frac{\partial L}{\partial w}
$$

donde:
- w: vector de pesos actuales  
- $\eta$: tasa de aprendizaje (learning rate)  
- $\frac{\partial L}{\partial w}$: gradiente de la pérdida respecto a los pesos  

En el caso estocástico, el gradiente se calcula sobre un **subconjunto (batch)** de los datos, lo que introduce cierta variabilidad pero acelera el entrenamiento.

---

## Interpretación geométrica
El gradiente $\frac{\partial L}{\partial w}$ apunta hacia la dirección de **mayor aumento** de la pérdida.  
Por tanto, el paso de actualización $ -\eta \frac{\partial L}{\partial w} $ mueve los parámetros **en sentido contrario**, buscando el mínimo local.

$$
\text{SGD step} = -\eta \cdot \text{gradient}
$$

Cada iteración reduce la pérdida L(w) hasta converger a un valor mínimo.

 11. Momentum Optimizer

In [ ]:
def momentum_step(
    w: np.ndarray,
    dw: np.ndarray,
    v: np.ndarray,
    lr: float,
    momentum: float
    )-> Tuple[np.ndarray, np.ndarray]:
    """
    Performs SGD with momentum update.
    Returns (w_new, v_new).
    """

    # Actualizamos la velocidad acumulando parte de la velocidad anterior
    # y sumando el gradiente actual.
    v_new = momentum * v + dw

    # Actualizamos los pesos usando la nueva velocidad.
    w_new = w - lr * v_new

    return w_new, v_new


En este ejercicio se implementa una actualización de pesos usando descenso por 
gradiente con momentum. La idea del momentum es acumular parte de las 
actualizaciones anteriores mediante una variable de velocidad. Esto permite que
el entrenamiento sea más estable, ya que el modelo no depende únicamente del 
gradiente calculado en el paso actual. Primero se actualiza la velocidad 
combinando la velocidad anterior con el gradiente actual, y luego se actualizan
 los pesos restando la tasa de aprendizaje multiplicada por esa nueva velocidad
 En términos prácticos, momentum ayuda a suavizar el movimiento de los
 parámetros durante el entrenamiento de la red neuronal.


# 15. Weight Initialization (Kaiming/He)

In [ ]:
def kaiming_init(shape: tuple) -> np.ndarray:
    """
    Kaiming/He normal initialization.
    """
    fan_in = shape[0]
    fan_out = shape[1]

    std = np.sqrt(2.0 / fan_in)

    weights = np.random.normal(
        loc=0.0,
        scale=std,
        size=shape
    )

    return weights
    pass

## Contexto
La **inicialización de Kaiming (He)** se diseñó específicamente para redes neuronales que usan la función de activación ReLU.  
Su objetivo es mantener la **varianza de las activaciones constante** a lo largo de las capas, evitando que los gradientes se desvanezcan o exploten.

---

## Derivación matemática

Consideremos una capa lineal con pesos W y entradas x:

$$
y = W \cdot x
$$

Queremos que la varianza de la salida y sea igual a la varianza de la entrada x:

$$
Var(y) = Var(x)
$$

Si los pesos W se inicializan con media cero y varianza Var(W), entonces:

$$
Var(y) = Var(W) \cdot Var(x) \cdot n
$$

donde n es el número de entradas (fan\_in).

Para mantener Var(y) = Var(x), se requiere:

$$
Var(W) = \frac{1}{n}
$$

Sin embargo, cuando se usa **ReLU**, aproximadamente la mitad de las activaciones se anulan (porque ReLU(x) = 0 para x < 0).  
Por tanto, la varianza efectiva se reduce a la mitad, y se compensa multiplicando por 2:

$$
Var(W) = \frac{2}{n}
$$

Esto se traduce en inicializar los pesos con una distribución normal de media 0 y desviación estándar:

$$
\sigma = \sqrt{\frac{2}{n}}
$$

# 17. Dropout Backward

In [ ]:
def dropout_backward(dout: np.ndarray, mask: Optional[np.ndarray], p: float, train: bool = True) -> np.ndarray:
    """
    Backward pass for inverted dropout.
    """
    if train:
        masked_grad = mask * dout
        dx = masked_grad / (1 - p)
    else:
        dx = dout

    return dx
    pass

## Contexto
El **Dropout** es una técnica de regularización que consiste en desactivar aleatoriamente un conjunto de neuronas durante el entrenamiento para evitar el sobreajuste (*overfitting*).  
Durante la propagación hacia atrás (**backpropagation**), los gradientes solo fluyen a través de las neuronas que **no fueron desactivadas**.

---

## Definición del proceso
Durante el **forward pass**, se aplica una máscara binaria m que indica qué neuronas se mantienen activas:

$$
y = \frac{m \cdot x}{1 - p}
$$

donde:
- x: entrada de la capa  
- m: máscara binaria (1 = activa, 0 = desactivada)  
- p: probabilidad de dropout  
- 1 - p: factor de escala para mantener la expectativa constante  

---

## Derivación del backward pass
En el **backward pass**, el gradiente de la pérdida L respecto a la entrada x se calcula como:

$$
\frac{\partial L}{\partial x} = \frac{m}{1 - p} \cdot \frac{\partial L}{\partial y}
$$

Esto significa que:
- Solo las neuronas activas (m = 1) reciben gradiente.  
- Las neuronas desactivadas (m = 0) no contribuyen al gradiente.  
- El factor $\frac{1}{1 - p}$ mantiene la escala esperada de las activaciones.

# 8.  Batch Normalization Forward

In [ ]:
def batchnorm_forward(
    x: np.ndarray,
    gamma: np.ndarray,
    beta: np.ndarray,
    eps: float = 1e-5,
    momentum: float = 0.9,
    running_mean: Optional[np.ndarray] = None,
    running_var: Optional[np.ndarray] = None,
    train: bool = True
    ):
   

    # Si no existen estadísticas acumuladas, se inicializan en cero.
    if running_mean is None:
        running_mean = np.zeros(x.shape[1])

    if running_var is None:
        running_var = np.zeros(x.shape[1])

    if train:
        # Media y varianza del batch actual, calculadas por columna.
        mean = np.mean(x, axis=0)
        var = np.mean((x - mean) ** 2, axis=0)

        # Normalización del batch.
        x_norm = (x - mean) / np.sqrt(var + eps)

        # Escalamiento y desplazamiento aprendibles.
        out = gamma * x_norm + beta

        # Actualización de estadísticas acumuladas.
        running_mean_new = momentum * running_mean + (1 - momentum) * mean
        running_var_new = momentum * running_var + (1 - momentum) * var

        # Guardamos información útil para backward.
        cache = {
            "x": x,
            "x_norm": x_norm,
            "mean": mean,
            "var": var,
            "gamma": gamma,
            "eps": eps
        }

    else:
        # En evaluación no usamos la media y varianza del batch,
        # sino las estadísticas acumuladas durante entrenamiento.
        x_norm = (x - running_mean) / np.sqrt(running_var + eps)

        out = gamma * x_norm + beta

        running_mean_new = running_mean
        running_var_new = running_var

        cache = {
            "x": x,
            "x_norm": x_norm,
            "mean": running_mean,
            "var": running_var,
            "gamma": gamma,
            "eps": eps
        }

    return out, cache, running_mean_new, running_var_new

En este ejercicio se implementa la pasada hacia adelante de Batch 
Normalization. Esta técnica normaliza las activaciones de una capa usando la 
 y la varianza del batch, con el objetivo de estabilizar el entrenamiento de 
 la red neuronal. Después de normalizar, se aplican dos parámetros aprendibles,
 gamma y beta, que permiten ajustar la escala y el desplazamiento de los datos 
 normalizados. Durante el entrenamiento se calculan las estadísticas del batch 
 actual y se actualizan estadísticas acumuladas. Durante evaluación, en cambio,
 se usan esas estadísticas acumuladas para mantener un comportamiento estable 
 del modelo.